<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA1/IL1.4/2-langsmith-evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Evaluación de Sistemas RAG con LangSmith

## Objetivos de Aprendizaje
- Comprender la importancia de la evaluación en sistemas RAG.
- Configurar LangSmith para trazabilidad y evaluación.
- Crear un dataset de evaluación con preguntas y respuestas de referencia.
- Ejecutar evaluadores automáticos para métricas como relevancia y fidelidad.
- Analizar los resultados de la evaluación para optimizar el sistema.

## ¿Qué es LangSmith?

LangSmith es una plataforma de LangChain para la observabilidad, el monitoreo y la evaluación de aplicaciones construidas con Modelos de Lenguaje Grandes (LLMs). Permite visualizar cada paso de una cadena o agente, analizar su rendimiento y evaluar la calidad de las respuestas de forma sistemática.

Para un sistema RAG, LangSmith nos ayuda a responder preguntas clave:
- **Recuperación (Retrieval)**: ¿Los documentos que encontramos son relevantes para la pregunta?
- **Generación (Generation)**: ¿La respuesta generada es fiel a los documentos recuperados? ¿Responde correctamente a la pregunta del usuario?

## 1. Instalación y Configuración

In [ ]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai langsmith openai python-dotenv


In [ ]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

import os
from openai import OpenAI
from langsmith import Client
import json

print("✅ Librerías importadas")

✅ Librerías importadas


### Configuración de Variables de Entorno

Para que LangSmith capture las trazas de nuestra aplicación necesitamos estas variables (ya están en el `.env` de la raíz del repo):

1. `LANGSMITH_TRACING`: se establece en `"true"` para activar la trazabilidad.
2. `LANGSMITH_API_KEY`: tu clave de LangSmith. La obtienes en [smith.langchain.com](https://smith.langchain.com).
3. `LANGSMITH_PROJECT`: el nombre del proyecto bajo el cual se agrupan las trazas. Muy útil para organizar el trabajo.
4. `LLM_API_KEY` y `LLM_BASE_URL`: la key de Mistral y su endpoint, para que el modelo de lenguaje responda.

> **Nota:** los nombres antiguos `LANGCHAIN_TRACING_V2`, `LANGCHAIN_API_KEY` y `LANGCHAIN_PROJECT` siguen funcionando, pero `LANGSMITH_*` es la nomenclatura actual.


In [ ]:
# Inicializar el cliente de LangSmith
client = Client()

print("✅ Variables de entorno y cliente de LangSmith configurados")

✅ Variables de entorno y cliente de LangSmith configurados


## 2. Sistema RAG Básico

Reutilizaremos el sistema RAG simple del notebook anterior. Este sistema utiliza una lista de documentos en memoria, una función de recuperación por palabras clave y un LLM para generar respuestas.

In [ ]:
# Base de documentos
documents = [
    "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.",
    "Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.",
    "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.",
    "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes.",
    "El prompt engineering es la práctica de diseñar instrucciones efectivas para obtener los mejores resultados de los modelos de IA."
]

# Cliente de chat: librería `openai` apuntada a Groq (patrón del proyecto)
def initialize_client():
    client = OpenAI(
        base_url=os.getenv("LLM_BASE_URL", "https://api.mistral.ai/v1"),
        api_key=os.getenv("LLM_API_KEY")
    )
    return client

openai_client = initialize_client()

print(f"📚 Base de datos con {len(documents)} documentos cargada.")
print("✅ Cliente de chat inicializado correctamente.")

📚 Base de datos con 5 documentos cargada.
✅ Cliente de chat inicializado correctamente.


In [ ]:
from langsmith.run_helpers import traceable

# Envolvemos las funciones con el decorador @traceable para que LangSmith las capture

@traceable(name="Recuperacion de Documentos")
def simple_retrieval(query, documents):
    relevant_docs = []
    query_lower = query.lower()
    for doc in documents:
        if any(word in doc.lower() for word in query_lower.split()):
            relevant_docs.append(doc)
    return relevant_docs[:3]

@traceable(name="Generacion de Respuesta")
def generate_response(client, query, context):
    prompt = f"""Contexto:
{context}

Pregunta: {query}

Responde basándote únicamente en el contexto proporcionado."""
    response = client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "mistral-small-latest"),
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content

@traceable(name="Pipeline RAG Completo")
def rag_pipeline(query):
    context_docs = simple_retrieval(query, documents)
    context = "".join(context_docs)
    answer = generate_response(openai_client, query, context)
    return {"answer": answer, "context": context_docs}

print("✅ Funciones del pipeline RAG definidas y trazables")

✅ Funciones del pipeline RAG definidas y trazables


### Prueba de Trazabilidad

Ahora, si ejecutamos nuestro pipeline, LangSmith registrará la ejecución completa, incluyendo los pasos intermedios que decoramos. Puedes ir a tu proyecto en LangSmith para ver la traza.

In [ ]:
resultado = rag_pipeline("¿Qué es RAG?")
print(resultado["answer"])

RAG (Retrieval-Augmented Generation) es un sistema de IA que combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.


## 3. Creación de un Dataset de Evaluación

Para evaluar nuestro sistema, necesitamos un "ground truth" (verdad fundamental), es decir, un conjunto de preguntas y las respuestas que consideramos correctas. En LangSmith, esto se gestiona a través de Datasets.

In [ ]:
dataset_name = "Dataset RAG Básico - Fundamentos IA"
description = "Preguntas y respuestas sobre conceptos básicos de IA para evaluar un RAG simple."

# Eliminar dataset si ya existe para evitar duplicados
try:
    existing_dataset = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=str(existing_dataset.id))
    print(f"🗑️ Dataset '{dataset_name}' existente eliminado.")
except Exception:
    pass # El dataset no existía, no hay nada que hacer

# Crear el nuevo dataset
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description=description,
)

print(f"✅ Dataset '{dataset_name}' creado.")

🗑️ Dataset 'Dataset RAG Básico - Fundamentos IA' existente eliminado.


✅ Dataset 'Dataset RAG Básico - Fundamentos IA' creado.


### Añadir Ejemplos al Dataset

Cada ejemplo consta de:
- `inputs`: Un diccionario con las entradas de nuestro sistema (en este caso, la `query`).
- `outputs`: Un diccionario con la salida de referencia que esperamos (la `answer` correcta).

In [ ]:
client.create_example(
    inputs={"query": "¿Qué es la inteligencia artificial?"},
    outputs={"answer": "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana."},
    dataset_id=dataset.id,
)

client.create_example(
    inputs={"query": "¿Para qué sirve LangChain?"},
    outputs={"answer": "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes."},
    dataset_id=dataset.id,
)

client.create_example(
    inputs={"query": "Explica qué es RAG"},
    outputs={"answer": "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas."},
    dataset_id=dataset.id,
)

print(f"✅ 3 ejemplos añadidos al dataset '{dataset_name}'.")

✅ 3 ejemplos añadidos al dataset 'Dataset RAG Básico - Fundamentos IA'.


## 4. Ejecución de la Evaluación

Ahora que tenemos el sistema y el dataset, podemos ejecutar la evaluación. LangSmith utilizará un LLM para comparar las respuestas generadas por nuestro `rag_pipeline` con las respuestas de referencia (`ground truth`) de nuestro dataset.

In [ ]:
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI

# --- El juez: un LLM que califica las respuestas del RAG ---
# Temperatura 0 para que la evaluación sea reproducible.
juez = ChatOpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL", "mistral-small-latest"),
    temperature=0,
)

# --- El evaluador ---
# La API actual de LangSmith espera una función con esta firma exacta:
#   (inputs, outputs, reference_outputs) -> {"key": ..., "score": ...}
# `inputs` y `reference_outputs` vienen del dataset; `outputs` es lo que
# devolvió la función evaluada.
def exactitud(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Compara la respuesta generada contra la respuesta de referencia."""
    prompt = f"""Eres un evaluador estricto. Determina si la RESPUESTA OBTENIDA
responde correctamente la PREGUNTA, usando la RESPUESTA ESPERADA como referencia.
No exijas coincidencia literal: evalúa si el contenido es equivalente.

PREGUNTA: {inputs["query"]}
RESPUESTA ESPERADA: {reference_outputs["answer"]}
RESPUESTA OBTENIDA: {outputs["answer"]}

Responde únicamente con una palabra: CORRECTO o INCORRECTO."""

    veredicto = juez.invoke(prompt).content.strip().upper()
    acierto = "CORRECTO" in veredicto and "INCORRECTO" not in veredicto
    return {"key": "exactitud", "score": 1.0 if acierto else 0.0}

# La función que será evaluada. Debe aceptar 'inputs' como un diccionario.
def target_function(inputs):
    return rag_pipeline(inputs["query"])

experiment_results = evaluate(
    target_function,                                  # La función a evaluar
    data=dataset_name,                                # Nuestro dataset en LangSmith
    evaluators=[exactitud],                           # Lista de evaluadores
    experiment_prefix="RAG Basico - Primera Prueba",  # Prefijo del experimento
    metadata={"version": "1.0.0"},                    # Metadatos para seguimiento
)

print("✅ Evaluación completada. Revisa los resultados en LangSmith.")

<ruta-local>:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'RAG Basico - Primera Prueba-e15e4a20' at:
https://smith.langchain.com/o/6ae35575-d843-42ae-af95-f4723becf432/datasets/acb863bb-dcc7-4b77-ace9-9b1dc997b6d9/compare?selectedSessions=0fd80abb-b85e-4808-814a-9dfc9352abb2




0it [00:00, ?it/s]

1it [00:02,  2.03s/it]

2it [00:03,  1.90s/it]

3it [00:05,  1.66s/it]

3it [00:05,  1.83s/it]

✅ Evaluación completada. Revisa los resultados en LangSmith.


## 5. Análisis de Resultados

Una vez completada la evaluación, puedes navegar a la pestaña **Experiments** en tu proyecto de LangSmith.

Allí encontrarás:
1. Un resumen del experimento con las puntuaciones medias de cada métrica.
2. Una tabla detallada con cada pregunta del dataset, la respuesta generada, la respuesta de referencia y las puntuaciones de la evaluación.
3. Para cada fila, puedes hacer clic para ver la traza completa y entender por qué el sistema respondió de esa manera (qué documentos recuperó, qué prompt se usó, etc.).

Este análisis te permite identificar puntos débiles. Por ejemplo:
- **Puntuaciones bajas de `correctness`**: Puede que la recuperación no esté funcionando bien o que el prompt de generación necesite ajustes.
- **Documentos irrelevantes en el contexto**: Indica que el método de `retrieval` debe ser mejorado (por ejemplo, pasando de búsqueda por palabras clave a búsqueda semántica con embeddings).

## Conclusión

La evaluación es un pilar fundamental en el desarrollo de sistemas de IA robustos. LangSmith nos ofrece un conjunto de herramientas poderosas para automatizar este proceso en sistemas RAG, permitiéndonos pasar de un desarrollo basado en la intuición a uno guiado por datos y métricas objetivas. Con este enfoque, podemos mejorar de forma iterativa la calidad y fiabilidad de nuestras aplicaciones.